In [ ]:

import pandas as pd
import numpy as np

# Se utiliza el Randomforest como aprendizaje supervisado, debido a que se quiere predecir el valor numero de las precipitaciones de los futuros meses en base a la data. 
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

df = pd.read_csv(
    "../data/processed/gatun_monthly.csv",
    parse_dates=["date"], # convierte la columna date a formato de fecha 0000-00-00 
    index_col="date" # utilizando las fechas como indice del DataFrame
).sort_index() # ordena cronologicamente antes de crear variables historicas 

# Variables conocidas antes del mes objetivo
df["lag_1"] = df["rainfall_mm"].shift(1) # Lluvia del mes anterior  
df["lag_2"] = df["rainfall_mm"].shift(2) # Lluvia hace dos meses 
df["lag_3"] = df["rainfall_mm"].shift(3) # Lluvia hace tres meses 
df["lag_12"] = df["rainfall_mm"].shift(12) # Lluvia el mismo mes pero hace un año (El mes que se quiere hacer la PREDICCIÓN)

# Estas variables permiten que Random Forest aprenda diferencias entre meses de la temporada seca y la lluviosa, siempre que esos patrones estén presentes en los datos.
# Representación circular de los meses 
df["month_sin"] = np.sin(2 * np.pi * df.index.month / 12)
df["month_cos"] = np.cos(2 * np.pi * df.index.month / 12)

# Aquí definimos las seis variables que utilizaremos para entrenar el modelo.
features = [
    "lag_1", "lag_2", "lag_3", "lag_12",
    "month_sin", "month_cos"
]

df = df.dropna(subset=features + ["rainfall_mm"])

# División cronológica
train = df.loc[: "2019-12-01"]
validation = df.loc["2020-01-01":"2022-12-01"]
test = df.loc["2023-01-01":"2025-12-01"]

model = RandomForestRegressor(
    n_estimators=300,
    min_samples_leaf=3,
    random_state=42,
    n_jobs=-1
)

model.fit(train[features], train["rainfall_mm"])

# Validación para ajustar el modelo
val_pred = model.predict(validation[features])

# Evaluación final sin ajustar sobre test
pred = model.predict(test[features])

# Modelo base: lluvia del mismo mes del año anterior
baseline = test["lag_12"]

for name, predictions in [
    ("Baseline estacional", baseline),
    ("Random Forest", pred)
]:
    mae = mean_absolute_error(test["rainfall_mm"], predictions)
    rmse = np.sqrt(
        mean_squared_error(test["rainfall_mm"], predictions)
    )
    print(f"{name}: MAE={mae:.2f} mm | RMSE={rmse:.2f} mm")


Baseline estacional: MAE=118.45 mm | RMSE=156.49 mm
Random Forest: MAE=72.35 mm | RMSE=99.41 mm


In [3]:

from sklearn.metrics import mean_absolute_error, mean_squared_error

# Se asume que train y test fueron creados
# previamente en el notebook de entrenamiento.

actual = test["rainfall_mm"]
rf_predictions = model.predict(test[features])
baseline_predictions = test["lag_12"]

results = pd.DataFrame({
    "Modelo": ["Baseline estacional", "Random Forest"],
    "MAE": [
        mean_absolute_error(actual, baseline_predictions),
        mean_absolute_error(actual, rf_predictions)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(actual, baseline_predictions)),
        np.sqrt(mean_squared_error(actual, rf_predictions))
    ]
})

print(results)


                Modelo         MAE        RMSE
0  Baseline estacional  118.450000  156.492560
1        Random Forest   72.350129   99.410632
